# DataScribes — DataProc PySpark Pipeline (Demo)

Run this notebook directly on the **NYU DataProc master node** via JupyterHub.  
All stages use PySpark natively — no `spark-submit` calls needed.

> **Demo mode:** runs on **10 datasets** (5 NYC + 5 Data.gov) by default — completes in ~5 minutes.  
> Change `NYC_LIMIT` and `DATA_GOV_LIMIT` to `100` in the config cell for the full 200-dataset production run.  
> Full 200-dataset results are already available in `data/generated_descriptions_all_200.csv`.

| Stage | What it does | Output |
|---|---|---|
| 1 | Ingestion & Sampling | HDFS metadata parquet |
| 2 | Profiling | HDFS profiles parquet |
| 3 | LLM Description Generation | HDFS descriptions parquet |
| 4 | Evaluation | Metrics summary |

**Prerequisites:** `ANTHROPIC_API_KEY` set in the config cell, `anthropic_pkg.zip` built in the Stage 3 setup cell.

---
## Configuration

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'findspark'], check=True)

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/temurin-11-jdk-amd64'
os.environ['SPARK_HOME'] = '/usr/lib/spark'
os.environ['PYSPARK_PYTHON'] = '/opt/conda/bin/python3'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/opt/conda/bin/python3'

import findspark
findspark.init('/usr/lib/spark')

In [ ]:
import os

# ── Edit these before running ──────────────────────────────────────────────────
NYU_NETID         = "at6370"
ANTHROPIC_API_KEY = "enter_api_key"
CLAUDE_MODEL      = "claude-sonnet-4-5"
ANTHROPIC_ZIP     = os.path.expanduser("~/anthropic_pkg.zip")

# Demo: 5 NYC + 5 Data.gov (10 total) — change to 100 for the full production run
NYC_LIMIT     = 5
DATA_GOV_LIMIT = 5
# ──────────────────────────────────────────────────────────────────────────────

HDFS_BASE = f"hdfs:///user/{NYU_NETID}_nyu_edu/data"
HDFS_META = f"{HDFS_BASE}/metadata/combined_metadata_with_samples_v2.parquet"
HDFS_PROF = f"{HDFS_BASE}/profiles/combined_profiles_spark_v2.parquet"
HDFS_DESC = f"{HDFS_BASE}/descriptions/generated_descriptions_sonnet.parquet"
LOCAL_OUT = os.path.expanduser("~/generated_descriptions_sonnet.parquet")

if not ANTHROPIC_API_KEY:
    print("WARNING: ANTHROPIC_API_KEY is not set — Stage 3 will fail.")
else:
    print("ANTHROPIC_API_KEY is set.")
print(f"HDFS base  : {HDFS_BASE}")
print(f"Demo mode  : {NYC_LIMIT} NYC + {DATA_GOV_LIMIT} Data.gov = {NYC_LIMIT + DATA_GOV_LIMIT} datasets")
print("Config OK.")

---
## Start SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import udf, col
import subprocess

spark = SparkSession.builder \
    .appName("DataScribes") \
    .master("local[2]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.submit.deployMode", "client") \
    .getOrCreate()
sc = spark.sparkContext

# Create HDFS base directory
subprocess.run(["hdfs", "dfs", "-mkdir", "-p", f"/user/{NYU_NETID}_nyu_edu/data"],
               capture_output=True)

print(f"Spark version : {spark.version}")
print(f"App name      : {sc.appName}")
print("SparkSession ready.")

---
## Clean HDFS outputs from previous runs

In [ ]:
import subprocess
for path in [HDFS_META, HDFS_PROF, HDFS_DESC]:
    subprocess.run(["hdfs", "dfs", "-rm", "-r", "-f", path], capture_output=True)
print("HDFS cleaned — starting fresh.")

---
## Stage 1 — Ingestion & Sampling

Fetches `NYC_LIMIT` + `DATA_GOV_LIMIT` datasets in parallel using Spark.  
Uses Socrata JSON API for NYC sample rows and streams CSVs for Data.gov.  
**Demo runtime: ~1–2 minutes. Full run (100+100): ~5–10 minutes.**

In [ ]:
import json, math, requests
from io import StringIO
import pandas as pd

NYC_CATALOG_URL        = "https://data.cityofnewyork.us/api/views.json"
NYC_VIEW_URL_TEMPLATE  = "https://data.cityofnewyork.us/api/views/{dataset_id}.json"
NYC_JSON_TEMPLATE      = "https://data.cityofnewyork.us/resource/{dataset_id}.json?$limit={limit}"
DATA_GOV_SEARCH_URL    = "https://catalog.data.gov/search"
HEADERS                = {"User-Agent": "Mozilla/5.0"}


def safe_get(d, k, default=None):
    return d[k] if k in d else default


def fetch_nyc_index(limit=100):
    resp = requests.get(NYC_CATALOG_URL, timeout=60, headers=HEADERS)
    resp.raise_for_status()
    records = []
    for entry in resp.json()[:limit]:
        did = safe_get(entry, "id")
        if not did: continue
        records.append({
            "dataset_id": str(did), "source": "nyc_open_data",
            "summary_json": json.dumps(entry, ensure_ascii=False),
            "download_url": f"https://data.cityofnewyork.us/api/views/{did}/rows.csv?accessType=DOWNLOAD",
        })
    return records


def fetch_data_gov_index(limit=100):
    resp = requests.get(DATA_GOV_SEARCH_URL, params={"q": "", "per_page": limit},
                        timeout=60, headers=HEADERS)
    resp.raise_for_status()
    records = []
    for entry in resp.json()["results"]:
        dcat = entry.get("dcat", {}) or {}
        did  = entry.get("identifier") or dcat.get("identifier")
        if not did: continue
        csv_url = None
        for dist in (dcat.get("distribution") or []):
            if not isinstance(dist, dict): continue
            url = dist.get("accessURL") or dist.get("downloadURL")
            mt  = (dist.get("mediaType") or "").lower()
            fmt = (dist.get("format") or "").lower()
            if url and ("csv" in mt or "csv" in fmt or str(url).lower().endswith(".csv")):
                csv_url = url; break
        records.append({
            "dataset_id": str(did), "source": "data_gov",
            "summary_json": json.dumps(entry, ensure_ascii=False),
            "download_url": csv_url,
        })
    return records


def fetch_nyc_json_sample(dataset_id, limit=5):
    url = NYC_JSON_TEMPLATE.format(dataset_id=dataset_id, limit=limit)
    try:
        r = requests.get(url, timeout=30, headers=HEADERS)
        r.raise_for_status()
        rows = r.json()
        return None, rows if isinstance(rows, list) else [], list(rows[0].keys()) if rows else []
    except Exception as e:
        return str(e), [], []


def fetch_csv_sample(url, limit=5):
    if not url: return None, [], []
    try:
        r = requests.get(url, timeout=30, stream=True, headers=HEADERS)
        r.raise_for_status()
        lines = []
        for line in r.iter_lines():
            if line: lines.append(line.decode("utf-8", errors="replace"))
            if len(lines) >= limit + 1: break
        r.close()
        if not lines: return None, [], []
        df = pd.read_csv(StringIO("\n".join(lines)))
        return None, df.to_dict(orient="records"), list(df.columns)
    except Exception as e:
        return str(e), [], []


def process_partition(rows):
    import json, requests
    from io import StringIO
    import pandas as pd

    HEADERS = {"User-Agent": "Mozilla/5.0"}
    NYC_VIEW_URL = "https://data.cityofnewyork.us/api/views/{dataset_id}.json"
    NYC_JSON_URL = "https://data.cityofnewyork.us/resource/{dataset_id}.json?$limit=5"

    def safe_get(d, k, default=None): return d[k] if k in d else default

    def nyc_sample(did):
        try:
            r = requests.get(NYC_JSON_URL.format(dataset_id=did), timeout=30, headers=HEADERS)
            r.raise_for_status()
            rows = r.json()
            if not isinstance(rows, list) or not rows: return None, [], []
            return None, rows, list(rows[0].keys())
        except Exception as e: return str(e), [], []

    def csv_sample(url):
        if not url: return None, [], []
        try:
            r = requests.get(url, timeout=30, stream=True, headers=HEADERS)
            r.raise_for_status()
            lines = []
            for line in r.iter_lines():
                if line: lines.append(line.decode("utf-8", errors="replace"))
                if len(lines) >= 6: break
            r.close()
            if not lines: return None, [], []
            df = pd.read_csv(StringIO("\n".join(lines)))
            return None, df.to_dict(orient="records"), list(df.columns)
        except Exception as e: return str(e), [], []

    out = []
    session = requests.Session()
    session.headers.update(HEADERS)

    for row in rows:
        did     = row["dataset_id"]
        source  = row["source"]
        summary = json.loads(row["summary_json"])
        dl_url  = row.get("download_url")

        try:
            if source == "nyc_open_data":
                detail = session.get(NYC_VIEW_URL.format(dataset_id=did), timeout=30).json()
                cols   = detail.get("columns", [])
                col_names  = [c.get("name", "") for c in cols]
                col_types  = [c.get("dataTypeName", "unknown") for c in cols]
                category   = safe_get(summary, "category")
                tags       = safe_get(summary, "tags", [])
                kws        = list(dict.fromkeys(([category] if category else []) + [t for t in tags if t]))
                license_i  = detail.get("license")
                license_n  = license_i.get("name") if isinstance(license_i, dict) else None
                err, s_rows, _ = nyc_sample(did)
                out.append({
                    "dataset_id": did, "source": source,
                    "title": safe_get(summary, "name"),
                    "original_description": safe_get(summary, "description"),
                    "keywords_json": json.dumps(kws),
                    "column_names_json": json.dumps(col_names),
                    "column_types_raw_json": json.dumps(col_types),
                    "download_url": dl_url,
                    "landing_page_url": f"https://data.cityofnewyork.us/d/{did}",
                    "record_count_estimate": None,
                    "last_updated": str(safe_get(summary, "rowsUpdatedAt", "")),
                    "license": license_n,
                    "sample_rows_json": json.dumps(s_rows) if s_rows else None,
                    "sample_error": err,
                    "raw_metadata_json": json.dumps({"summary_entry": summary, "detail_entry": detail}),
                })
            else:
                dcat  = summary.get("dcat", {}) or {}
                tags  = summary.get("keyword") or dcat.get("keyword") or []
                title = summary.get("title") or dcat.get("title")
                desc  = summary.get("description") or dcat.get("description")
                err, s_rows, s_cols = csv_sample(dl_url)
                out.append({
                    "dataset_id": did, "source": source,
                    "title": title, "original_description": desc,
                    "keywords_json": json.dumps([str(x) for x in tags] if isinstance(tags, list) else []),
                    "column_names_json": json.dumps(s_cols),
                    "column_types_raw_json": json.dumps([]),
                    "download_url": dl_url,
                    "landing_page_url": dcat.get("landingPage"),
                    "record_count_estimate": None,
                    "last_updated": str(dcat.get("modified", "")),
                    "license": str(dcat.get("license", "")) or None,
                    "sample_rows_json": json.dumps(s_rows) if s_rows else None,
                    "sample_error": err,
                    "raw_metadata_json": json.dumps(summary),
                })
        except Exception as e:
            out.append({
                "dataset_id": did, "source": source,
                "title": None, "original_description": None,
                "keywords_json": "[]", "column_names_json": "[]",
                "column_types_raw_json": "[]", "download_url": dl_url,
                "landing_page_url": None, "record_count_estimate": None,
                "last_updated": None, "license": None,
                "sample_rows_json": None,
                "sample_error": f"partition_error: {str(e)}",
                "raw_metadata_json": row["summary_json"],
            })
    return iter(out)


print("Stage 1 functions defined.")

In [ ]:
print(f"Fetching {NYC_LIMIT} NYC datasets...")
nyc_index = fetch_nyc_index(limit=NYC_LIMIT)
print(f"  {len(nyc_index)} NYC datasets fetched")

print(f"Fetching {DATA_GOV_LIMIT} Data.gov datasets...")
gov_index = fetch_data_gov_index(limit=DATA_GOV_LIMIT)
print(f"  {len(gov_index)} Data.gov datasets fetched")

combined   = nyc_index + gov_index
partitions = max(10, math.ceil(len(combined) / 2))
print(f"\nTotal: {len(combined)} datasets across {partitions} partitions")

schema_s1 = StructType([
    StructField("dataset_id",            StringType(), True),
    StructField("source",                StringType(), True),
    StructField("title",                 StringType(), True),
    StructField("original_description",  StringType(), True),
    StructField("keywords_json",         StringType(), True),
    StructField("column_names_json",     StringType(), True),
    StructField("column_types_raw_json", StringType(), True),
    StructField("download_url",          StringType(), True),
    StructField("landing_page_url",      StringType(), True),
    StructField("record_count_estimate", StringType(), True),
    StructField("last_updated",          StringType(), True),
    StructField("license",               StringType(), True),
    StructField("sample_rows_json",      StringType(), True),
    StructField("sample_error",          StringType(), True),
    StructField("raw_metadata_json",     StringType(), True),
])

rdd_s1 = sc.parallelize(combined, partitions).mapPartitions(process_partition)
df_s1  = spark.createDataFrame(rdd_s1, schema=schema_s1)
df_s1.write.mode("overwrite").parquet(HDFS_META)

print("\nStage 1 COMPLETE.")
df_s1.groupBy("source").count().show(truncate=False)

---
## Stage 2 — Profiling

Reads metadata parquet from HDFS, computes per-column stats via Spark UDFs.  
**Expected runtime: ~2–5 minutes.**

In [ ]:
import json

def _safe_load_rows(sample_rows_json):
    if sample_rows_json is None: return []
    try:
        obj = json.loads(sample_rows_json)
        return obj if isinstance(obj, list) else []
    except Exception: return []


def get_sample_row_count(sample_rows_json):
    return len(_safe_load_rows(sample_rows_json))


def get_sample_col_count(sample_rows_json):
    rows = _safe_load_rows(sample_rows_json)
    if not rows: return 0
    return len(rows[0].keys()) if isinstance(rows[0], dict) else 0


def build_sample_profile_json(sample_rows_json):
    rows = _safe_load_rows(sample_rows_json)
    if not rows or not isinstance(rows[0], dict): return json.dumps({})
    profile = {}
    for c in rows[0].keys():
        vals    = [str(r.get(c)) for r in rows if r.get(c) is not None and str(r.get(c)).strip()]
        missing = sum(1 for r in rows if r.get(c) is None or str(r.get(c)).strip() == "")
        numeric = all(_is_numeric(v) for v in vals) if vals else False
        profile[c] = {
            "missing_count":       missing,
            "unique_count_sample": len(set(vals)),
            "example_values":      vals[:3],
            "inferred_sample_type": "numeric_like" if numeric else "text_like",
        }
    return json.dumps(profile)


def _is_numeric(s):
    try: float(s); return True
    except Exception: return False


row_count_udf  = udf(get_sample_row_count,    IntegerType())
col_count_udf  = udf(get_sample_col_count,    IntegerType())
profile_udf    = udf(build_sample_profile_json, StringType())

df_s2 = spark.read.parquet(HDFS_META)
df_s2 = (
    df_s2
    .withColumn("sample_row_count",          row_count_udf(col("sample_rows_json")))
    .withColumn("sample_column_count",        col_count_udf(col("sample_rows_json")))
    .withColumn("sample_columns_profile_json", profile_udf(col("sample_rows_json")))
)

df_s2.write.mode("overwrite").parquet(HDFS_PROF)

print("Stage 2 COMPLETE.")
df_s2.select("source", "title", "sample_row_count", "sample_column_count").show(10, truncate=False)

---
## Stage 3 — LLM Description Generation

Calls `claude-sonnet-4-5` via the Anthropic API using `mapPartitions`.  
The `anthropic` package is distributed to workers via `sc.addPyFile()`.  
**Demo runtime: ~1–2 minutes. Full run (200 datasets): ~10–15 minutes.**

### Setup — bundle the anthropic package (run once)

In [ ]:
import subprocess, os

pkg_dir = os.path.expanduser("~/anthropic_pkg")

# Install into local folder
subprocess.run(["pip", "install", "anthropic", "-t", pkg_dir, "-q", "--exists-action", "i"], check=True, capture_output=True)
print(f"anthropic installed into {pkg_dir}")

# Zip from inside the folder so structure is flat
subprocess.run(f"cd {pkg_dir} && zip -r {ANTHROPIC_ZIP} .", shell=True, check=True)
print(f"Zipped to {ANTHROPIC_ZIP}")

# Verify
result = subprocess.run(["unzip", "-l", ANTHROPIC_ZIP], capture_output=True, text=True)
print("\n".join(result.stdout.splitlines()[:5]))

In [ ]:
import time

if not ANTHROPIC_API_KEY:
    raise ValueError("Set ANTHROPIC_API_KEY in the Configuration cell before running Stage 3.")

# Distribute anthropic package to all workers
api_key_bc = sc.broadcast(ANTHROPIC_API_KEY)
model_bc   = sc.broadcast(CLAUDE_MODEL)


def build_prompt(row):
    import json
    title        = row.title or "Untitled Dataset"
    source       = row.source or ""
    orig_desc    = (row.original_description or "").strip()[:500]
    keywords     = json.loads(row.keywords_json or "[]")
    col_names    = json.loads(row.column_names_json or "[]")
    sample_rows  = json.loads(row.sample_rows_json or "[]") if row.sample_rows_json else []
    profile      = json.loads(row.sample_columns_profile_json or "{}") if row.sample_columns_profile_json else {}

    lines = [f"Dataset Title: {title}", f"Source: {source}"]
    if orig_desc:   lines.append(f"Original Description: {orig_desc}")
    if keywords:    lines.append(f"Keywords: {', '.join(keywords[:8])}")
    if col_names:   lines.append(f"Columns ({len(col_names)} total): {', '.join(col_names[:12])}")
    if profile:
        plines = []
        for cn, stats in list(profile.items())[:6]:
            ex  = stats.get("example_values", [])[:2]
            inf = stats.get("inferred_sample_type", "")
            plines.append(f"  - {cn} ({inf}): e.g. {', '.join(str(v) for v in ex)}")
        if plines: lines.append("Column Profiles:\n" + "\n".join(plines))
    if sample_rows and isinstance(sample_rows, list):
        lines.append(f"Sample row: {json.dumps(sample_rows[0], ensure_ascii=False)[:300]}")

    return (
        "You are a data catalog assistant. Write a clear, concise description "
        "(3-5 sentences) for the following dataset.\n"
        "The description should help a data analyst quickly understand what the "
        "dataset contains, what it can be used for, and any notable characteristics.\n"
        "Do not copy the original description verbatim. Be specific and informative.\n\n"
        + "\n".join(lines) + "\n\nDescription:"
    )


def generate_for_partition(rows):
    import sys, glob, time
    for p in glob.glob('/opt/conda/lib/python3*/site-packages'):
        if p not in sys.path:
            sys.path.insert(0, p)
    import anthropic
    client = anthropic.Anthropic(api_key=api_key_bc.value)
    out = []
    for row in rows:
        try:
            prompt = build_prompt(row)
            msg = client.messages.create(
                model=model_bc.value, max_tokens=300,
                messages=[{"role": "user", "content": prompt}]
            )
            desc, err = msg.content[0].text.strip(), None
        except Exception as e:
            desc, err = None, str(e)
            time.sleep(2)
        out.append({
            "dataset_id":            row.dataset_id,
            "source":                row.source,
            "title":                 row.title,
            "original_description":  row.original_description,
            "generated_description": desc,
            "generation_model":      model_bc.value,
            "generation_error":      err,
        })
    return iter(out)


schema_s3 = StructType([
    StructField("dataset_id",            StringType(), True),
    StructField("source",                StringType(), True),
    StructField("title",                 StringType(), True),
    StructField("original_description",  StringType(), True),
    StructField("generated_description", StringType(), True),
    StructField("generation_model",      StringType(), True),
    StructField("generation_error",      StringType(), True),
])

df_s3_in = spark.read.parquet(HDFS_PROF).repartition(2)
rdd_s3   = df_s3_in.rdd.mapPartitions(generate_for_partition)
df_s3    = spark.createDataFrame(rdd_s3, schema=schema_s3)
df_s3.write.mode("overwrite").parquet(HDFS_DESC)

print("Stage 3 COMPLETE.")
df_s3.groupBy("source", "generation_model").count().show(truncate=False)

---
## Download Output from HDFS

In [ ]:
import subprocess, os

result = subprocess.run(
    ["hdfs", "dfs", "-get", "-f", HDFS_DESC, LOCAL_OUT],
    capture_output=True, text=True
)
print(result.stdout or result.stderr or "Download complete.")
print(f"Files in {LOCAL_OUT}:")
for f in sorted(os.listdir(LOCAL_OUT)):
    print(f"  {f}")

---
## Convert Parquet → CSV

Merges the 10 part files into a single CSV required by the standalone evaluation scripts.

In [ ]:
import pandas as pd

df_out   = pd.read_parquet(LOCAL_OUT)
csv_path = os.path.expanduser("~/generated_descriptions_all_200.csv")
df_out.to_csv(csv_path, index=False)

n_total   = len(df_out)
n_success = df_out["generated_description"].notna().sum()
print(f"Saved {n_total} rows to {csv_path}")
print(f"Successful generations: {n_success}/{n_total} ({100*n_success/n_total:.1f}%)")

---
## Stage 4 — Evaluation

| Metric group | Metrics |
|---|---|
| Automatic | Success rate, novelty score, title coverage, word/sentence counts |
| Text similarity | ROUGE-1/2/L, METEOR, BERTScore F1 |
| Retrieval | TF-IDF cosine ranking + NDCG@5 and NDCG@10 |

**Expected runtime: ~5 minutes (BERTScore downloads a model on first run).**

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "rouge-score", "bert-score", "nltk", "scikit-learn"], check=True)
print("Eval dependencies ready.")

In [ ]:
df_out[["title", "generation_error"]].to_string()

In [ ]:
import re, math

STOPWORDS = {"the", "a", "an", "of", "in", "and", "or", "for", "to", "by", "on", "at", "with"}

def word_set(text):
    if not text or (isinstance(text, float) and math.isnan(text)): return set()
    return set(re.findall(r"[a-z]+", str(text).lower()))

def word_count(text):
    if not text or (isinstance(text, float) and math.isnan(text)): return 0
    return len(str(text).split())

def sentence_count(text):
    if not text or (isinstance(text, float) and math.isnan(text)): return 0
    return len(re.findall(r"[.!?]+", str(text).strip()))

def novelty_score(gen, orig):
    g, o = word_set(gen), word_set(orig)
    return round(len(g - o) / len(g), 3) if g else 0.0

def title_coverage(gen, title):
    if not gen or not title: return False
    tw = {w for w in word_set(title) if w not in STOPWORDS and len(w) > 2}
    return bool(tw & word_set(gen))


eval_df = df_out.copy()
eval_df["success"]            = eval_df["generated_description"].notna()
eval_df["gen_word_count"]     = eval_df["generated_description"].apply(word_count)
eval_df["gen_sentence_count"] = eval_df["generated_description"].apply(sentence_count)
eval_df["follows_3_5"]        = eval_df["gen_sentence_count"].between(3, 5)
eval_df["novelty_score"]      = eval_df.apply(lambda r: novelty_score(r["generated_description"], r["original_description"]), axis=1)
eval_df["title_coverage"]     = eval_df.apply(lambda r: title_coverage(r["generated_description"], r["title"]), axis=1)

ok = eval_df[eval_df["success"]]
print(f"Success rate   : {eval_df['success'].mean()*100:.1f}%")
print(f"Avg words      : {ok['gen_word_count'].mean():.1f}")
print(f"Avg sentences  : {ok['gen_sentence_count'].mean():.2f}")
print(f"Follows 3-5s   : {ok['follows_3_5'].mean()*100:.1f}%")
print(f"Novelty score  : {ok['novelty_score'].mean():.3f}")
print(f"Title coverage : {ok['title_coverage'].mean()*100:.1f}%")

In [ ]:
import nltk
from rouge_score import rouge_scorer as rouge_lib
from nltk.translate import meteor_score as meteor_lib
from nltk.tokenize import word_tokenize

for r in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    nltk.download(r, quiet=True)

scorer   = rouge_lib.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
scorable = eval_df[
    eval_df["generated_description"].notna() &
    eval_df["original_description"].notna() &
    (eval_df["original_description"].str.strip() != "")
].copy()

r1, r2, rL, meteors = [], [], [], []
for _, row in scorable.iterrows():
    ref = str(row["original_description"]).strip()
    hyp = str(row["generated_description"]).strip()
    s   = scorer.score(ref, hyp)
    r1.append(s["rouge1"].fmeasure)
    r2.append(s["rouge2"].fmeasure)
    rL.append(s["rougeL"].fmeasure)
    try:    m = meteor_lib.meteor_score([word_tokenize(ref.lower())], word_tokenize(hyp.lower()))
    except: m = 0.0
    meteors.append(m)

scorable["rouge1"] = r1
scorable["rouge2"] = r2
scorable["rougeL"] = rL
scorable["meteor"] = meteors

print(f"ROUGE-1 : {scorable['rouge1'].mean():.4f}")
print(f"ROUGE-2 : {scorable['rouge2'].mean():.4f}")
print(f"ROUGE-L : {scorable['rougeL'].mean():.4f}")
print(f"METEOR  : {scorable['meteor'].mean():.4f}")

In [ ]:
from bert_score import score as bertscore_fn

print("Running BERTScore (downloads model on first run — ~2 min)...")
P, R, F1 = bertscore_fn(
    scorable["generated_description"].tolist(),
    scorable["original_description"].tolist(),
    lang="en", verbose=True
)
scorable["bertscore_f1"] = F1.numpy()
print(f"BERTScore F1 : {scorable['bertscore_f1'].mean():.4f}")

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

QUERIES = [
    ("NYC taxi trip data",           ["nyc", "taxi", "trip", "cab", "fare", "ride"]),
    ("weather wind speed dataset",   ["weather", "wind", "speed", "temperature", "climate"]),
    ("housing property values",      ["housing", "property", "real estate", "home", "price"]),
    ("crime incident reports",       ["crime", "incident", "arrest", "complaint", "police"]),
    ("public health disease data",   ["health", "disease", "hospital", "covid", "vaccination"]),
    ("school education enrollment",  ["school", "education", "student", "enrollment", "grade"]),
    ("traffic collision accidents",  ["traffic", "collision", "accident", "crash", "vehicle"]),
    ("restaurant food inspection",   ["restaurant", "food", "inspection", "violation", "health"]),
]

def dcg(rels):  return sum(r / math.log2(i+2) for i, r in enumerate(rels))
def ndcg(rels): ideal = sorted(rels, reverse=True); d = dcg(ideal); return dcg(rels)/d if d else 0.0
def kw_rel(text, kws): return int(any(k in text.lower() for k in kws))

def rank_corpus(query, corpus):
    vec = TfidfVectorizer(stop_words="english")
    try:
        mat  = vec.fit_transform(corpus + [query])
        sims = cos_sim(mat[-1], mat[:-1]).flatten()
        return sorted(range(len(corpus)), key=lambda i: sims[i], reverse=True)
    except Exception: return list(range(len(corpus)))

ok_df = df_out[df_out["generated_description"].notna()].copy()
orig_corpus = [f"{r['title'] or ''} {r['original_description'] or ''}" for _, r in ok_df.iterrows()]
gen_corpus  = [f"{r['title'] or ''} {r['generated_description'] or ''}" for _, r in ok_df.iterrows()]

ret_records = []
for query, keywords in QUERIES:
    orig_ranked = rank_corpus(query, orig_corpus)
    gen_ranked  = rank_corpus(query, gen_corpus)
    for k in [5, 10]:
        orig_rels = [kw_rel(orig_corpus[i], keywords) for i in orig_ranked[:k]]
        gen_rels  = [kw_rel(gen_corpus[i],  keywords) for i in gen_ranked[:k]]
        ret_records.append({
            "query": query, "k": k,
            "ndcg_original":  round(ndcg(orig_rels), 4),
            "ndcg_generated": round(ndcg(gen_rels),  4),
            "delta":          round(ndcg(gen_rels) - ndcg(orig_rels), 4),
        })

ret_df = pd.DataFrame(ret_records)
at10   = ret_df[ret_df["k"] == 10]
print(f"NDCG@10 original  : {at10['ndcg_original'].mean():.4f}")
print(f"NDCG@10 generated : {at10['ndcg_generated'].mean():.4f}")
print(f"NDCG@10 delta     : {at10['delta'].mean():+.4f}")
at10[["query", "ndcg_original", "ndcg_generated", "delta"]]

---
## Final Summary

In [ ]:
print("=" * 65)
print("DATASCRIBES — DATAPROC PIPELINE RESULTS")
print("=" * 65)

print(f"\n Datasets processed : {n_total}")
print(f" Successful         : {n_success} ({100*n_success/n_total:.1f}%)")
print(f" Failed             : {n_total - n_success}")
print(f" Model              : {CLAUDE_MODEL}")

print("\n— Automatic Metrics —")
print(f" Avg words         : {ok['gen_word_count'].mean():.1f}")
print(f" Avg sentences     : {ok['gen_sentence_count'].mean():.2f}")
print(f" Follows 3-5s      : {ok['follows_3_5'].mean()*100:.1f}%")
print(f" Novelty score     : {ok['novelty_score'].mean():.3f}")
print(f" Title coverage    : {ok['title_coverage'].mean()*100:.1f}%")

print("\n— Text Similarity (vs. original) —")
print(f" ROUGE-1     : {scorable['rouge1'].mean():.4f}")
print(f" ROUGE-2     : {scorable['rouge2'].mean():.4f}")
print(f" ROUGE-L     : {scorable['rougeL'].mean():.4f}")
print(f" METEOR      : {scorable['meteor'].mean():.4f}")
print(f" BERTScore F1: {scorable['bertscore_f1'].mean():.4f}")

print("\n— Retrieval NDCG@10 —")
print(f" Original  : {at10['ndcg_original'].mean():.4f}")
print(f" Generated : {at10['ndcg_generated'].mean():.4f}")
print(f" Delta     : {at10['delta'].mean():+.4f}")
print("=" * 65)

In [ ]:
spark.stop()
print("SparkSession stopped.")